# Gemma: Diabetes — single-dataset fine-tuning

Historical experiment source: `bench_gemma/FewZeroShot/LLM_Gemma3_4B_Fine_tune_with_Diabetes_(ep=20).ipynb`. Outputs and stale result commentary were removed for publication. Scientific logic is retained; these experiments and their reported results have not been rerun or validated here.

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm
import math


In [ ]:
import os
from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# Настройка промпта для Diabetes
prompt_config = {
        "task": "Predict whether a patient has diabetes",
        "pos_label": "yes",
        "neg_label": "no",
        "entity": "Patient",
        "question": "Does this patient have diabetes?"
    }

output_dir = "./outputs/gemma_diabetes"

# 1. Загрузка данных

In [ ]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset():
    path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database") + "/diabetes.csv"
    df = pd.read_csv(path)
    target_name = 'Outcome'
    X = df.drop(columns=[target_name])
    y = df[target_name]

    feature_names = X.columns.to_list()

    le = LabelEncoder()
    df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):
    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset()
train_df, val_df, test_df = split_dataset(df, target_name)

In [ ]:
df.info()

In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")

# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    pos = prompt_config['pos_label'].lower()
    neg = prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos:
        return 1
    elif response == neg:
        return 0

    # Начинается с метки
    if response.startswith(pos):
        return 1
    elif response.startswith(neg):
        return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words:
        return 1
    elif neg in words:
        return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    pr  = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return roc, f1, acc, pr, rec

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name, prompt_config)
print(f"\nПример текста:\n{sample_text[:300]}")

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Загрузка модели Gemma 3 4B Instruct
model_name = "google/gemma-3-4b-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Загрузка базовой модели
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    used_mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU total memory: {total_mem:.2f} GB")
    print(f"GPU allocated memory: {used_mem:.2f} GB")

In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['pos_label'] if ex[target_name] == 1 else prompt_config['neg_label']

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import gc
import torch.nn.functional as F

BATCH_SIZE_EVAL = 32

def batched_rows(df, batch_size):
    for start in range(0, len(df), batch_size):
        yield df.iloc[start:start + batch_size]

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def predict_batch_with_prob(prompts, prompt_config, model, tokenizer, device, max_new_tokens=3):
    model_inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    input_len = model_inputs.input_ids.shape[1]
    del model_inputs

    generated_ids = outputs.sequences[:, input_len:]
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    responses = [resp.strip().lower() for resp in responses]
    del generated_ids

    first_token_logits = outputs.scores[0]
    del outputs

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]
    selected_logits = torch.stack([first_token_logits[:, pos_id], first_token_logits[:, neg_id]], dim=1)
    probs = F.softmax(selected_logits, dim=1)
    prob_pos = probs[:, 0].detach().cpu().tolist()
    del first_token_logits, selected_logits, probs

    flush_gpu()
    return responses, prob_pos

# Тест на маленьком батче
test_prompts = [
    create_prompt(test_df.iloc[i], feature_names, target_name, prompt_config, tokenizer)
    for i in range(min(2, len(test_df)))
]
responses, probs = predict_batch_with_prob(test_prompts, prompt_config, base_model, tokenizer, device)
for i, (response, prob) in enumerate(zip(responses, probs)):
    print(f"Sample {i}: response='{response}', probability={prob:.4f}")

# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
# @title
df_0 = train_df[train_df[target_name] == 0]
df_1 = train_df[train_df[target_name] == 1]

if len(df_0) > len(df_1):
    df_majority = df_0
    df_minority = df_1
else:
    df_majority = df_1
    df_minority = df_0

seed = 42
print(f"До балансировки:")
print(f"{prompt_config['neg_label']}:  {len(df_majority)}")
print(f"{prompt_config['pos_label']}: {len(df_minority)}")

# Oversample to match
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=seed
)

train_df_balanced = pd.concat([df_majority, df_minority_upsampled])
train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

print(f"\nПосле балансировки:")
print(train_df_balanced[target_name].value_counts())

In [ ]:
# Создание текстов для обучения

def create_dataset(df):
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )
    texts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Подготовка данных"):
        input_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        target = prompt_config['pos_label']  if row[target_name] == 1 else prompt_config['neg_label']

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts

train_texts = create_dataset(train_df_balanced)
train_dataset = Dataset.from_dict({"text": train_texts})

In [ ]:
# Токенизация
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
        return_token_type_ids=True
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

# Для Gemma training loop ожидает token_type_ids в батче.
if "token_type_ids" not in tokenized_dataset.column_names:
    tokenized_dataset = tokenized_dataset.map(
        lambda batch: {"token_type_ids": [[0] * len(ids) for ids in batch["input_ids"]]},
        batched=True,
    )

print(f"\n Подготовлено {len(tokenized_dataset)} примеров для обучения")

In [ ]:
num_epochs = 20
batch_size = 32

# Загрузка базовой модели
model_lora = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

model_lora.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Настройка LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

# Аргументы для обучения
training_args_lora = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    tf32=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_torch_fused",
    warmup_steps=50,
    max_grad_norm=1.0,
    weight_decay=0.01,
    report_to="none",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    group_by_length=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    torch_compile=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Обучение
print(f"\nНачинаем обучение на {num_epochs} эпох")

train_start_time = time.time()
trainer_lora.train()
training_time = time.time() - train_start_time

print(f"Обучение завершено за {training_time/60:.1f} минут")

trainer_lora.save_model(output_dir)
print(f"Модель сохранена в {output_dir}")


In [ ]:
import glob
import re
from peft import PeftModel
print("\nОценка checkpoint на val")

# Поиск и сортировка checkpoint
checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
checkpoints_sorted = sorted(
    checkpoints,
    key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
)

print(f"\nНайдено {len(checkpoints_sorted)} checkpoints\n")

# Хранение результатов
results_list = []
best_auc = 0.0
best_checkpoint = None

# Оценка каждого checkpoint
for checkpoint_path in checkpoints_sorted:
    checkpoint_name = checkpoint_path.split('/')[-1]
    checkpoint_num = int(re.search(r'checkpoint-(\d+)', checkpoint_name).group(1))

    # Загрузка модели
    model_eval = PeftModel.from_pretrained(base_model, checkpoint_path)
    model_eval.eval()

    # Оценка
    y_true, y_pred, y_prob = [], [], []
    n_batches = math.ceil(len(val_df) / BATCH_SIZE_EVAL)
    for batch_df in tqdm(batched_rows(val_df, BATCH_SIZE_EVAL), total=n_batches, desc=checkpoint_name):
        prompts = [
            create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
            for _, row in batch_df.iterrows()
        ]
        responses, probs = predict_batch_with_prob(prompts, prompt_config, model_eval, tokenizer, device)

        for (_, row), response, prob in zip(batch_df.iterrows(), responses, probs):
            prediction = parse_prediction(response, prompt_config)
            y_true.append(row[target_name])
            y_pred.append(prediction)
            y_prob.append(prob)

        del prompts, responses, probs

    # Метрики
    roc, f1, acc, pr, rec = compute_metrics(y_true, y_pred, y_prob)

    results_list.append({
        'Checkpoint': checkpoint_num,
        'ROC-AUC': roc,
        'F1': f1,
        'Accuracy': acc,
        'Precision': pr,
        'Recall': rec
    })

    if roc > best_auc:
        best_auc = roc
        best_checkpoint = checkpoint_path

    del model_eval
    torch.cuda.empty_cache()

In [ ]:
print(f"\nЛучший checkpoint: {best_checkpoint}")

In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('Checkpoint')
results_df

In [ ]:
model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint)
model_finetuned.eval()

In [ ]:
# Проверка качества обучения Fine-tuned модели

test_samples = {
    prompt_config['pos_label']: test_df[test_df[target_name] == 1].sample(n=10, random_state=42),
    prompt_config['neg_label']: test_df[test_df[target_name] == 0].sample(n=10, random_state=42)
}

# Проверка распределения вероятностей
print("\nРаспределение вероятностей")
sample_prompts = [
    create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    for _, row in test_df.sample(n=100, random_state=42).iterrows()
]
all_probs = []
for i in range(0, len(sample_prompts), BATCH_SIZE_EVAL):
    batch = sample_prompts[i:i + BATCH_SIZE_EVAL]
    _, batch_probs = predict_batch_with_prob(batch, prompt_config, model_finetuned, tokenizer, device)
    all_probs.extend(batch_probs)

all_probs = np.array(all_probs)
print(f"P({prompt_config['pos_label']}) - Mean: {all_probs.mean():.4f}, Std: {all_probs.std():.4f}")
print(f"P({prompt_config['pos_label']}) - Min: {all_probs.min():.4f}, Max: {all_probs.max():.4f}")
print(f"P({prompt_config['pos_label']}) - Median: {np.median(all_probs):.4f}")

In [ ]:
# Оценка на тестовой выборке (batched)
y_true_fine, y_pred_fine, y_prob_fine = [], [], []

n_batches = math.ceil(len(test_df) / BATCH_SIZE_EVAL)
start_time = time.time()

for batch_df in tqdm(batched_rows(test_df, BATCH_SIZE_EVAL), total=n_batches, desc="Test evaluation"):
    prompts = [
        create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
        for _, row in batch_df.iterrows()
    ]
    responses, probs = predict_batch_with_prob(prompts, prompt_config, model_finetuned, tokenizer, device)

    for (_, row), response, prob in zip(batch_df.iterrows(), responses, probs):
        prediction = parse_prediction(response, prompt_config)
        y_true_fine.append(row[target_name])
        y_pred_fine.append(prediction)
        y_prob_fine.append(prob)

    del prompts, responses, probs

fine_tuned_time = time.time() - start_time

print(f"fine_tuned_time: {fine_tuned_time:.1f}s")
print(f"per sample: {fine_tuned_time / len(y_true_fine):.4f}s")
if torch.cuda.is_available():
    print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# Вычисляем метрики

roc_fine, f1_fine, acc_fine, pr_fine, rec_fine = compute_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine)
)
print("Результаты fine-tune:")
print(f"ROC AUC: {roc_fine}")
print(f"F1 Score: {f1_fine}")
print(f"Accuracy: {acc_fine}")
print(f"Precision: {pr_fine}")
print(f"Recall: {rec_fine}")


In [ ]:
fine_tuned_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine),
    n_iter=1000
)

print("\nРезультаты fine-tune (bootstrap метрики с доверительными интервалами):")
for key, value in fine_tuned_metrics_bootstrap.items():
    print(f"  {key}: {value}")